# The Convolution Operation

**Companion lesson:** https://ml-viz.vercel.app/courses/cnns/01-convolution-operation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## 2D Convolution from scratch

In [ ]:
def conv2d(X, K, stride=1, pad=0):
    X = np.pad(X, pad)
    H, W = X.shape
    FH, FW = K.shape
    OH = (H - FH) // stride + 1
    OW = (W - FW) // stride + 1
    out = np.zeros((OH, OW))
    for i in range(OH):
        for j in range(OW):
            out[i, j] = np.sum(X[i*stride:i*stride+FH, j*stride:j*stride+FW] * K)
    return out

# Create a test image
img = np.zeros((10, 10))
img[3:7, 3:7] = 1  # white square
img[1:3, 1:3] = 0.5

# Edge detection kernels
edge_h = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]])
edge_v = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(img, cmap='magma')
axes[0].set_title('Input Image', color='white')
axes[1].imshow(conv2d(img, edge_h, pad=1), cmap='RdBu_r')
axes[1].set_title('Horizontal Edges', color='white')
axes[2].imshow(conv2d(img, edge_v, pad=1), cmap='RdBu_r')
axes[2].set_title('Vertical Edges', color='white')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

## A convolution you can check by hand

The 10x10 demo above isn't traceable by eye. Here is the lesson's small example: a 4x4 input with the vertical-edge (Sobel) filter, every sliding-window dot product printed, plus the parameter-count breakdown (weights + bias).

In [ ]:
Xh = np.array([[1, 2, 3, 0],
               [0, 1, 2, 3],
               [3, 0, 1, 2],
               [2, 3, 0, 1]])
Kh = np.array([[1, 0, -1],
               [2, 0, -2],
               [1, 0, -1]])     # vertical-edge (Sobel)

f = Kh.shape[0]
O = Xh.shape[0] - f + 1          # valid, stride 1 -> 4-3+1 = 2
out = np.zeros((O, O), dtype=int)
for i in range(O):
    for j in range(O):
        patch = Xh[i:i+f, j:j+f]
        out[i, j] = int(np.sum(patch * Kh))
        print(f'pos ({i},{j}): sum(patch * K) = {out[i, j]}')
print('\nfeature map:\n', out)
print('matches [[-4,-4],[4,-4]]:', out.tolist() == [[-4, -4], [4, -4]])

# Parameter count broken out: weights + biases
C_in, C_out, k = 3, 64, 3
weights = C_in * C_out * k * k
biases = C_out
print(f'\nconv 3->64, 3x3:  weights {C_in}*{C_out}*{k}*{k} = {weights}  + biases {biases}  = {weights + biases}')


## Effect of stride and padding

In [ ]:
img_large = np.random.randn(8, 8)
kernel = np.random.randn(3, 3)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
titles = ['Valid (pad=0, stride=1)', 'Same (pad=1, stride=1)', 'Stride=2']
configs = [(0, 1), (1, 1), (0, 2)]
for ax, (p, s), t in zip(axes, configs, titles):
    out = conv2d(img_large, kernel, stride=s, pad=p)
    ax.imshow(out, cmap='magma')
    ax.set_title(f'{t}\nOutput: {out.shape}', color='white', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Output size and multi-channel convolution

Output side length: $O = \lfloor (W - F + 2P)/S \rfloor + 1$ for input $W$, filter $F$, padding $P$, stride $S$. A conv layer sums over **all input channels** and has one bias per output filter.

In [ ]:
def conv_out(W, F, P, S):
    return (W - F + 2 * P) // S + 1

for W, F, P, S in [(32, 3, 1, 1), (32, 3, 0, 1), (32, 5, 2, 2), (28, 3, 0, 1)]:
    print(f'W={W} F={F} P={P} S={S} -> {conv_out(W, F, P, S)}')

# Params of a conv layer: (F*F*C_in + 1) * C_out
F, C_in, C_out = 3, 3, 64
print('conv params:', (F * F * C_in + 1) * C_out)

## Key takeaways

- Convolution slides a small **filter** over the input, computing local weighted sums (feature maps).
- **Weight sharing** + **translation equivariance** make CNNs efficient and shift-robust.
- Output size $= \lfloor (W-F+2P)/S \rfloor + 1$; padding preserves spatial size.
- A filter spans **all input channels**; the layer learns `C_out` such filters.